# GCP Pose Estimation — Kaggle Training

## Before running: download the data as a ZIP and upload to Kaggle

**1. Download the Drive folder as ZIP (on your laptop)**
- Open [drive.google.com](https://drive.google.com) → find the shared GCP data folder
- Right-click the folder → **Download**
- Chrome will zip it and download. If the folder is >2 GB, Drive splits it into multiple ZIPs — download **all** of them.

**2. Upload the ZIP(s) as a Kaggle dataset**
- kaggle.com → your profile → **Datasets** → **New Dataset**
- Upload all ZIP file(s) → title: `skylark-gcp-data` → visibility: **Private** → Create

**3. Add that dataset as input to this notebook**
- In this notebook → **Add Input** (top right) → Your Datasets → `skylark-gcp-data` → Add

Then run cells top to bottom. GPU T4×2 + Internet must both be ON.

In [ ]:
# 1. Clone repo + install dependencies
!git clone https://github.com/J4KE-B/skylark-gcp gcp
%cd gcp
!pip -q install -r requirements.txt

In [ ]:
# 2. Wire config to Kaggle input dataset
import os, json, yaml, glob
from pathlib import Path

hits = sorted(glob.glob('/kaggle/input/**/gcp_marks.json', recursive=True), key=len)
assert hits, 'gcp_marks.json not found — add skylark-gcp-data as notebook input'
label_path = hits[0]
BASE = os.path.dirname(label_path)
print('Dataset root:', BASE)
print('Contents:', sorted(os.listdir(BASE)))

# Load + patch labels first; we use a real label key to locate the image root
os.makedirs('data', exist_ok=True)
raw = json.load(open(label_path))
patched = {k.replace('&', 'and'): v for k, v in raw.items()}
local_labels = 'data/gcp_marks.json'
json.dump(patched, open(local_labels, 'w'))
n_fixed = sum(1 for k in raw if '&' in k)
if n_fixed: print(f'Patched {n_fixed} label keys (& -> and)')

def find_dir(base, *names):
    for n in names:
        p = os.path.join(base, n)
        if os.path.isdir(p): return p
    raise FileNotFoundError(f'None of {names} in {base}')

def resolve_by_label(base, label_keys):
    """Find the dir that actually holds the labelled site folders, regardless of wrapper
    levels (train_clean/train/train_dataset/...) or junk dirs (__MACOSX). Locates the site
    folder of a sample label key and returns its parent."""
    site = next(iter(label_keys)).split('/')[0]
    for p in Path(base).rglob(site):
        if p.is_dir() and '__MACOSX' not in p.parts:
            return str(p.parent)
    return base

def descend_to_images(base, max_depth=8):
    """Test set (no labels): descend, skipping junk, until a folder holds images or branches."""
    junk = {'__MACOSX', '.ipynb_checkpoints'}
    d = base
    for _ in range(max_depth):
        entries = [e for e in os.listdir(d) if e not in junk]
        subs = [e for e in entries if os.path.isdir(os.path.join(d, e))]
        has_img = any(f.lower().endswith(('.jpg', '.jpeg')) for f in entries)
        if has_img or len(subs) != 1:
            return d
        d = os.path.join(d, subs[0])
    return d

train_base = find_dir(BASE, 'train_clean', 'train_dataset', 'train')
test_base  = find_dir(BASE, 'test_clean',  'test_dataset',  'test')
train_dir  = resolve_by_label(train_base, patched)
test_dir   = descend_to_images(test_base)
print(f'train_dir : {train_dir}')
print(f'test_dir  : {test_dir}')

cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['paths'] = {'label_file': local_labels, 'train_dir': train_dir,
                'test_dir': test_dir, 'output_dir': 'outputs'}
yaml.safe_dump(cfg, open('configs/config.yaml', 'w'))
print('Config updated.')

labels = json.load(open(local_labels))
found  = sum(1 for k in labels if os.path.exists(os.path.join(train_dir, k)))
pct    = 100 * found // len(labels)
print(f'Train images on disk: {found}/{len(labels)} ({pct}%)')
print('OK — enough to train.' if found >= len(labels) * 0.8 else
      'WARNING: <80% images — check folder names above.')

In [ ]:
# 3. SMOKE: 3 epochs, then inspect predictions schema before committing GPU time
import yaml
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 3
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import json; p = json.load(open('predictions.json'))
k = next(iter(p)); print(k, '->', p[k]); print('total:', len(p))

In [ ]:
# 4. FULL run: 60 epochs (early-stops on patience=15)
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 60
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml

In [ ]:
# 5. Final predictions + download artifacts
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import pandas as pd; pd.read_csv('outputs/training_log.csv').tail()

In [ ]:
# 6. Visualize predictions at ORIGINAL resolution — hollow circle, split into parts
import os, json, glob, cv2, shutil, zipfile, math

pred_hits = sorted(glob.glob('/kaggle/working/**/predictions.json', recursive=True), key=len)
assert pred_hits, 'predictions.json not found — run the predict cell first'
preds = json.load(open(pred_hits[0]))
print(f'Loaded {len(preds)} predictions from {pred_hits[0]}')

def descend_to_images(base, max_depth=8):
    junk = {'__MACOSX', '.ipynb_checkpoints'}
    d = base
    for _ in range(max_depth):
        entries = [e for e in os.listdir(d) if e not in junk]
        subs = [e for e in entries if os.path.isdir(os.path.join(d, e))]
        has_img = any(f.lower().endswith(('.jpg', '.jpeg')) for f in entries)
        if has_img or len(subs) != 1:
            return d
        d = os.path.join(d, subs[0])
    return d

test_base = sorted(glob.glob('/kaggle/input/**/test_clean', recursive=True) +
                   glob.glob('/kaggle/input/**/test_dataset', recursive=True), key=len)[0]
test_dir = descend_to_images(test_base)
print('test_dir:', test_dir)

out_dir = '/kaggle/working/viz'
shutil.rmtree(out_dir, ignore_errors=True); os.makedirs(out_dir)

drawn = 0
for key, val in preds.items():
    p = os.path.join(test_dir, key)
    if not os.path.exists(p): continue
    img = cv2.imread(p)
    if img is None: continue
    h, w = img.shape[:2]
    x, y = int(val['mark']['x']), int(val['mark']['y'])
    s = val['verified_shape']
    r  = max(int(w * 0.03),  70)
    th = max(int(w * 0.003), 3)
    fs = max(w / 600.0, 1.5)
    cv2.circle(img, (x, y), r, (0, 0, 255), th)
    cv2.putText(img, f'{s} ({x},{y})', (x + r + 8, y - r - 8),
                cv2.FONT_HERSHEY_SIMPLEX, fs, (0, 255, 255), th)
    cv2.imwrite(os.path.join(out_dir, key.replace('/', '_')), img,
                [cv2.IMWRITE_JPEG_QUALITY, 100])
    drawn += 1

# Split the annotated images into PARTS so each zip is downloadable
PARTS = 4                                   # bump to 6/8 if each part is still too big
files = sorted(glob.glob(f'{out_dir}/*'))
per = math.ceil(len(files) / PARTS)
print(f'\nAnnotated {drawn} images -> splitting into {PARTS} zips:')
for i in range(PARTS):
    chunk = files[i*per:(i+1)*per]
    if not chunk: break
    zp = f'/kaggle/working/viz_part{i+1}.zip'
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_STORED) as z:  # JPEGs already compressed
        for f in chunk:
            z.write(f, os.path.basename(f))
    print(f'  {zp}: {len(chunk)} imgs, {os.path.getsize(zp)/1e6:.0f} MB')

## Artifacts to download

From the Kaggle Output panel (right sidebar), download these three files:

- `gcp/outputs/best.pt` — model weights (upload to Drive, copy the shareable link)
- `gcp/outputs/training_log.csv` — open it, read the last row for PCK@10/25/50 + Macro-F1
- `gcp/predictions.json` — this is your submission file for Skylark

Then update `README.md` in the repo:
1. Fill in the Results table with the PCK/F1 numbers
2. Add the Drive link for `best.pt`
3. `git add README.md && git commit -m "docs: add training results and weights link" && git push`